# Dispatch Lab：A* + Genetic Algorithm 端到端 Demo

這份 notebook 直接使用網頁應用的現行 `scheduler` 引擎。流程是：載入 costmap → 建立配送訂單 → 計算 A* 距離／時間矩陣 → 產生優先權重 → 執行多世界 GA → 檢視收斂與無碰撞路徑。

In [ ]:
from pathlib import Path
import json, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'app.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from PIL import Image
import matplotlib.pyplot as plt
from scheduler.core import Costmap, Order, prepare, weights, Objective, evolve

## 1. 地圖與訂單
黑色像素是障礙；機器人半徑會先膨脹障礙，A* 不允許斜切牆角。

In [ ]:
meta = json.loads((ROOT / 'data/map/hotel.json').read_text())
grid = Costmap(Image.open(ROOT / 'data/map/hotel.png'), resolution=0.1, radius=0.2)
orders = [
    Order(f'Room {101 + i}', x, y, importance=5 + i % 5, waited=20 * i, patience=300)
    for i, (x, y) in enumerate(meta['points'])
]
matrix = prepare(grid, orders, tuple(meta['start']))
print(f'{len(orders)} orders; {len(matrix["paths"])} directed A* paths prepared')

## 2. 權重與基因演算法
每張訂單會依重要度、等待、風險、距離及目的地附近速度得到 sigmoid 權重。GA 只排列中間配送點，起點固定，完成後回到起點。

In [ ]:
priority_rows = weights(orders, matrix, cruise=0.8)
objective = Objective(orders, matrix, priority_rows, return_to_start=True)
result = evolve(
    objective, population=40, generations=40, worlds=4,
    sower=True, interval=10, seed=42
)
print('Visit order:', [stop['id'] for stop in result['best']['schedule']])
print('Distance (m):', round(result['best']['distance'], 2))
print('Duration (s):', round(result['best']['duration'], 2))
print('Sower migrations:', result['events'])

## 3. 收斂與實際 A* 路徑
左圖比較各世界的最佳 loss；右圖依最佳訪問順序串起真正的 A* 網格路徑。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for world in range(4):
    axes[0].plot(
        [row['generation'] for row in result['history']],
        [row['worlds'][world]['loss'] for row in result['history']],
        label=f'World {world + 1}',
    )
axes[0].set(xlabel='Generation', ylabel='Loss', title='GA convergence')
axes[0].legend()

axes[1].imshow(grid.pixels, cmap='gray', vmin=0, vmax=255)
route = result['best']['route']
for start, end in zip(route, route[1:]):
    path = matrix['paths'][f'{start}:{end}']
    axes[1].plot([p[0] for p in path], [p[1] for p in path], color='seagreen')
for stop in result['best']['schedule']:
    axes[1].annotate(str(stop['position']), matrix['points'][stop['node']], color='darkorange')
axes[1].set_title('Collision-free delivery route')
plt.tight_layout()
plt.show()